# System-Level EV Auxiliary Thermal Interaction Model

## 1. Motivation
## 2. Scenario Definition
## 3. Model Overview
## 4. Assumptions
## 5. Governing Equations
## 6. Numerical Implementation
## 7. Results
## 8. Discussion
## 9. Limitations and Future Work

1. Motivation

Electrified vehicle range is often discussed in terms of battery capacity and drive cycle efficiency. However, auxiliary thermal loads — particularly cabin conditioning and battery thermal management — can meaningfully influence real-world energy consumption, especially under extreme ambient conditions.

In battery electric and range-extended electric architectures, thermal systems are no longer isolated subsystems. Cabin cooling, battery heat rejection, compressor control strategy, and ambient conditions are tightly coupled. Decisions in one domain often influence performance in another.

One example of this coupling is compressor derate logic. In many architectures, compressor output may be reduced beyond a certain battery temperature threshold to protect durability limits. While protective in intent, this strategy can introduce trade-offs:

Reduced cooling capacity under high ambient load

Changes in cabin cooldown behavior

Altered auxiliary energy draw

Perceived range impact

In real-world operation, auxiliary thermal loads are rarely steady from key-on. Following hot-soak conditions, the initial cabin cooldown phase can dominate compressor energy consumption for several minutes before approaching steady operation. This transient behavior can disproportionately influence perceived range and energy efficiency.

The purpose of this notebook is not to replicate detailed OEM multiphysics simulations, but to construct a simplified, transparent system-level model that helps visualize these trade-offs. By isolating key variables and applying reasonable engineering approximations, we aim to better understand how auxiliary thermal strategies influence vehicle-level energy behavior.

Simplified models, when clearly framed, can provide architectural insight even without full system fidelity.

## 2. Scenario Definition

This notebook simulates a hot-soak cabin cooldown event under steady driving conditions to evaluate the transient impact of auxiliary thermal loads on vehicle energy consumption.

### Baseline Scenario (Editable Parameters)

- Ambient temperature, $T_{amb}$ : 40°C  
- Cabin air initial temperature, $T_{a,0}$ : 60°C (hot soak)  
- Cabin interior mass initial temperature, $T_{i,0}$ : 60°C  
- Cabin setpoint temperature, $T_{set}$ : 22°C  
- Battery initial temperature, $T_{b,0}$ : 45°C  
- Simulation duration: 20 minutes  
- Time step: 1–5 seconds  
- Drive condition: steady load (constant current)  
- Cabin control strategy: bang-bang cooling (maximum cooling until near setpoint)

This scenario reflects a common real-world condition where a vehicle parked under high ambient temperatures undergoes rapid cabin cooldown after key-on.

## 3. Model Overview

To study the transient interaction between cabin cooling, battery thermal behavior, and auxiliary power draw, the vehicle thermal system is represented using a simplified system-level model.

The model captures the key components that influence auxiliary thermal energy consumption during a cabin cooldown event.

### System Nodes

The system is represented by three primary thermal nodes:

| Node | Description |
|-----|-------------|
| $T_a$ | Cabin air temperature |
| $T_i$ | Cabin interior thermal mass temperature (seats, panels, dashboard) |
| $T_b$ | Battery temperature |

### Heat Transfer Interactions

The following thermal interactions are modeled:

1. **Ambient heat gain into the cabin**
   - Heat transfer from ambient air to cabin air.

2. **Interior-to-air heat exchange**
   - Thermal exchange between cabin interior materials and cabin air.

3. **Cabin cooling**
   - HVAC system removes heat from cabin air during cooldown.

4. **Battery heat generation**
   - Battery generates heat proportional to electrical current ($I^2R$).

5. **Battery heat rejection**
   - Heat removed from the battery through a simplified cooling path.

### HVAC Power Consumption

The compressor power required to support cooling loads is approximated using a coefficient of performance (COP) relationship:

$$
P_{comp} = \frac{Q_{cool}}{COP(T_{amb})}
$$

where:

- $Q_{cool}$ represents the total thermal load served by the HVAC system.
- $COP(T_{amb})$ is modeled as a piecewise function of ambient temperature.

### Control Strategy

During the transient cabin cooldown phase, the HVAC system operates in **bang-bang control mode**:

- Maximum cooling capacity is applied when cabin temperature exceeds the setpoint.
- Cooling is reduced once the cabin approaches the target temperature.

Cabin cooling is prioritized during the transient simulation period.

### Modeling Philosophy

The purpose of this model is not to reproduce detailed OEM simulations, but rather to create a **transparent system-level framework** that highlights key thermal and energy trade-offs.

Simplified physics-based approximations are used so that the interactions between cabin cooling demand, battery thermal behavior, and compressor power consumption can be visualized clearly.

## 4. Assumptions

To maintain transparency and interpretability, several simplifying assumptions are made in this model. The goal is to capture the dominant system-level interactions while keeping the simulation lightweight and easy to understand.

### Thermal Modeling Assumptions

1. **Lumped Thermal Mass Representation**

   The cabin is represented using two thermal masses:

   - Cabin air temperature: $T_a$
   - Cabin interior mass temperature: $T_i$

   The interior mass represents seats, panels, dashboard, and other materials that store thermal energy.

2. **Battery as a Lumped Thermal Mass**

   The battery is modeled as a single thermal node with temperature $T_b$.  
   Spatial temperature gradients within the battery pack are not modeled.

3. **Battery Heat Generation**

   Battery heat generation is approximated using:

   $$
   Q_{gen} = I^2 R_{int}
   $$

   where:
   - $I$ is the drive current (assumed constant)
   - $R_{int}$ is an effective internal resistance.

4. **Simplified Heat Rejection from Battery**

   Battery cooling is represented using a simplified linear heat rejection model:

   $$
   Q_{b,cool} = k_b (T_b - T_{amb})
   $$

   This term approximates the combined effect of coolant circulation and heat exchanger performance.

5. **HVAC Cooling Capacity**

   Cabin cooling capacity is limited by a maximum available cooling rate $Q_{cool,max}$.

6. **Coefficient of Performance (COP)**

   Compressor efficiency is modeled using a simplified **piecewise relationship with ambient temperature**:

   $$
   COP = COP(T_{amb})
   $$

   This reflects the degradation in HVAC efficiency at higher ambient temperatures.

7. **Cabin Control Strategy**

   During the cabin cooldown phase, a **bang-bang control strategy** is used:

   - Maximum cooling is applied when cabin temperature exceeds the setpoint.
   - Cooling demand decreases once the cabin approaches the target temperature.

8. **Excluded Phenomena**

   The following effects are intentionally excluded in Version 1 of the model:

   - Solar radiation load
   - Air infiltration and leakage
   - Detailed refrigerant cycle modeling
   - Multi-zone cabin airflow distribution
   - Dynamic drive-cycle variation

These simplifications allow the model to remain transparent while still capturing key transient thermal interactions.

## 5. Governing Equations

This section summarizes the simplified governing equations used to simulate transient cabin cooldown, battery temperature response, and resulting compressor power demand.

### 5.1 Cabin Thermal Model (Two-Lump: Air + Interior)

**Ambient-to-air heat gain**
$$
Q_{amb \rightarrow a} = U_a A_a (T_{amb} - T_a)
$$

**Interior-to-air heat exchange**
$$
Q_{i \leftrightarrow a} = h_{ia} A_{ia} (T_i - T_a)
$$

**Cabin air energy balance**
$$
C_a \frac{dT_a}{dt} = Q_{amb \rightarrow a} + Q_{i \leftrightarrow a} - Q_{cool}
$$

**Interior thermal mass energy balance**
$$
C_i \frac{dT_i}{dt} = -Q_{i \leftrightarrow a}
$$

where $C_a = m_a c_p$ and $C_i = m_i c_p$.

---

### 5.2 Battery Thermal Model (Lumped + $I^2R$)

**Heat generation**
$$
Q_{gen} = I^2 R_{int}
$$

**Heat rejection (simplified)**
$$
Q_{b,cool} = k_b (T_b - T_{amb})
$$

**Battery energy balance**
$$
C_b \frac{dT_b}{dt} = Q_{gen} - Q_{b,cool}
$$

where $C_b = m_b c_p$.

---

### 5.3 Cabin Control (Bang-Bang Cooling)

During the cabin cooldown phase, the HVAC system applies maximum cooling while the cabin air temperature exceeds the setpoint.

A simple representation is:

- If $T_a > T_{set}$:
  $$
  Q_{cool} = Q_{cool,max}
  $$
- Else:
  $$
  Q_{cool} = Q_{cool,maint}
  $$

For Version 1, $Q_{cool,maint}$ can be set to 0 (or a small non-zero value if desired).

---

### 5.4 COP and Compressor Power

The compressor electrical power is computed from the total thermal load served by the HVAC system:

$$
P_{comp} = \frac{Q_{cool} + Q_{b,cool}}{COP(T_{amb})}
$$

$COP(T_{amb})$ is modeled using a simple piecewise function to reflect reduced efficiency at higher ambient temperatures.

## 6. Numerical Implementation

The system is simulated using a simple forward-Euler time integration scheme with a fixed time step $\Delta t$.

For each time step:

1. Compute heat transfer terms:
   - $Q_{amb \rightarrow a}$
   - $Q_{i \leftrightarrow a}$
2. Apply bang-bang cooling to compute $Q_{cool}$
3. Compute battery heat generation $Q_{gen}$ and rejection $Q_{b,cool}$
4. Update temperatures using Euler integration:

$$
T(t+\Delta t)=T(t)+\Delta t\cdot \frac{dT}{dt}
$$

The compressor power is computed at each time step as:

$$
P_{comp}(t) = \frac{Q_{cool}(t) + Q_{b,cool}(t)}{COP(T_{amb})}
$$

Finally, cumulative auxiliary energy is computed by integrating compressor power over time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1) Scenario inputs (editable)
# -----------------------------
T_amb = 40.0          # degC
T_set = 22.0          # degC

T_a0 = 60.0           # cabin air initial (degC)
T_i0 = 60.0           # interior mass initial (degC)
T_b0 = 45.0           # battery initial (degC)

t_end_min = 20.0      # minutes
dt = 2.0              # seconds (use 1-5 sec; 2 sec is a nice default)

# ---------------------------------
# 2) Thermal model parameters (v1)
# ---------------------------------
cp = 1005.0  # J/(kg*K)  (approx air Cp; used as a generic Cp for simplicity)

# Cabin air capacitance (effective)
m_air_eff = 2.5            # kg (effective cabin air mass)
C_a = m_air_eff * cp       # J/K

# Interior thermal mass capacitance (effective)
m_int_eff = 120.0          # kg (lumped interior mass)
C_i = m_int_eff * 900.0    # J/K (use ~900 J/kgK as a generic interior Cp)

# Ambient -> cabin air heat gain
UA_a = 120.0               # W/K  (lumped conductance)

# Interior <-> air coupling
hA_ia = 180.0              # W/K  (lumped coupling)

# Cabin cooling (bang-bang)
Q_cool_max = 5000.0        # W (max cabin cooling capacity)
Q_cool_maint = 0.0         # W (your choice A)

# Battery parameters (lumped)
C_b = 200000.0             # J/K (effective battery thermal capacitance)
R_int = 0.010              # ohm (effective internal resistance)
I_drive = 150.0            # A (constant current assumption)
k_b = 40.0                 # W/K (battery heat rejection to ambient proxy)

# -----------------------------------------
# 3) Piecewise COP model (simple, editable)
# -----------------------------------------
def COP(Tamb):
    # Piecewise approximation (illustrative, not OEM-specific)
    # Higher COP at moderate ambient, degrades at high ambient.
    if Tamb <= 25:
        return 3.0
    elif Tamb <= 40:
        # linearly drop from 3.0 at 25C to 2.2 at 40C
        return 3.0 - (Tamb - 25.0) * (0.8 / 15.0)
    else:
        # beyond 40C, drop more quickly to a floor
        return max(1.6, 2.2 - (Tamb - 40.0) * 0.05)

# -----------------------------
# 4) Time vector + storage
# -----------------------------
t_end = t_end_min * 60.0
t = np.arange(0, t_end + dt, dt)

T_a = np.zeros_like(t)
T_i = np.zeros_like(t)
T_b = np.zeros_like(t)

Q_cool = np.zeros_like(t)
P_comp = np.zeros_like(t)
E_comp_Wh = np.zeros_like(t)  # cumulative energy

# initial conditions
T_a[0] = T_a0
T_i[0] = T_i0
T_b[0] = T_b0

# -----------------------------
# 5) Euler simulation loop
# -----------------------------
for k in range(len(t) - 1):
    # Heat flows
    Q_amb_to_a = UA_a * (T_amb - T_a[k])
    Q_i_to_a = hA_ia * (T_i[k] - T_a[k])

    # Bang-bang cabin cooling
    if T_a[k] > T_set:
        Q_cool[k] = Q_cool_max
    else:
        Q_cool[k] = Q_cool_maint

    # Battery heat generation and rejection
    Q_gen = (I_drive ** 2) * R_int
    Q_b_cool = k_b * (T_b[k] - T_amb)

    # ODEs
    dTa_dt = (Q_amb_to_a + Q_i_to_a - Q_cool[k]) / C_a
    dTi_dt = (-Q_i_to_a) / C_i
    dTb_dt = (Q_gen - Q_b_cool) / C_b

    # Update states
    T_a[k + 1] = T_a[k] + dt * dTa_dt
    T_i[k + 1] = T_i[k] + dt * dTi_dt
    T_b[k + 1] = T_b[k] + dt * dTb_dt

    # Compressor power and cumulative energy
    P_comp[k] = (Q_cool[k] + Q_b_cool) / COP(T_amb)  # W
    E_comp_Wh[k + 1] = E_comp_Wh[k] + (P_comp[k] * dt) / 3600.0

# fill last sample for plotting
Q_cool[-1] = Q_cool[-2]
P_comp[-1] = P_comp[-2]

# -----------------------------
# 6) Plots (minimum viable set)
# -----------------------------
plt.figure()
plt.plot(t/60.0, T_a, label="Cabin air $T_a$")
plt.plot(t/60.0, T_i, label="Interior mass $T_i$")
plt.axhline(T_set, linestyle="--", label="Setpoint $T_{set}$")
plt.xlabel("Time (min)")
plt.ylabel("Temperature (°C)")
plt.title("Cabin Cooldown Transient")
plt.legend()
plt.show()

plt.figure()
plt.plot(t/60.0, T_b, label="Battery $T_b$")
plt.axhline(T_amb, linestyle="--", label="Ambient $T_{amb}$")
plt.xlabel("Time (min)")
plt.ylabel("Temperature (°C)")
plt.title("Battery Temperature Response")
plt.legend()
plt.show()

plt.figure()
plt.plot(t/60.0, P_comp/1000.0, label="Compressor Power")
plt.xlabel("Time (min)")
plt.ylabel("Power (kW)")
plt.title("Compressor Electrical Power (Approx.)")
plt.legend()
plt.show()

plt.figure()
plt.plot(t/60.0, E_comp_Wh/1000.0, label="Cumulative Compressor Energy")
plt.xlabel("Time (min)")
plt.ylabel("Energy (kWh)")
plt.title("Cumulative Auxiliary Energy (Compressor)")
plt.legend()
plt.show()

print(f"Total compressor energy over {t_end_min:.0f} min: {E_comp_Wh[-1]/1000.0:.3f} kWh")

## 7. Results

Using the baseline hot-soak scenario ($T_{amb}=40^\circ C$, $T_{a,0}=60^\circ C$, $T_{set}=22^\circ C$), the model produces the following transient behavior:

- Cabin air temperature reaches the setpoint in approximately **11.5–12 minutes** under bang-bang cooling.
- Compressor power remains elevated during the cabin cooldown phase, then drops close to zero once the setpoint is met (since $Q_{cool,maint}=0$ in Version 1).
- Total compressor energy consumed over a 20-minute window is approximately **0.71 kWh**.

These results highlight that a significant fraction of auxiliary thermal energy is consumed during the initial transient cabin cooldown period.

## 8. Discussion (System-Level Takeaways)

Even in this simplified model, several system-level observations stand out:

1. **Transient dominates auxiliary energy consumption**
   - The majority of compressor energy is consumed during the initial cabin cooldown window (first ~12 minutes). Once the setpoint is met, auxiliary energy draw drops significantly.

2. **Interior thermal mass is a key driver of cooldown behavior**
   - The interior node ($T_i$) cools more slowly than cabin air ($T_a$), continuing to transfer heat back into the air. This delays the approach to steady behavior and increases compressor runtime.

3. **Energy impact is sensitive to scenario, not just components**
   - Hot-soak initial conditions can create a high auxiliary energy demand event even under a steady drive condition. This helps explain why real-world efficiency perception can vary strongly with parking and ambient conditions.

4. **Why this matters for EV energy architecture**
   - Although OEM tools may model these effects with higher fidelity, a simplified model like this can still help visualize the structure of the trade-off: cabin comfort response vs auxiliary energy consumption under transient conditions.

Future versions of this notebook can explore sensitivity to ambient temperature, cooling capacity, COP degradation, and battery thermal coupling / derate strategies.

In [ ]:
def run_simulation(T_amb):

    # initial conditions
    T_a = T_a0
    T_i = T_i0
    T_b = T_b0

    energy_Wh = 0
    setpoint_time = None

    for k in range(len(t)-1):

        # heat flows
        Q_amb_to_a = UA_a * (T_amb - T_a)
        Q_i_to_a = hA_ia * (T_i - T_a)

        # bang-bang cabin cooling
        if T_a > T_set:
            Q_cool = Q_cool_max
        else:
            Q_cool = Q_cool_maint

            if setpoint_time is None:
                setpoint_time = t[k]/60.0

        # battery heat
        Q_gen = (I_drive**2) * R_int
        Q_b_cool = k_b * (T_b - T_amb)

        # temperature derivatives
        dTa_dt = (Q_amb_to_a + Q_i_to_a - Q_cool) / C_a
        dTi_dt = (-Q_i_to_a) / C_i
        dTb_dt = (Q_gen - Q_b_cool) / C_b

        # update states
        T_a += dt * dTa_dt
        T_i += dt * dTi_dt
        T_b += dt * dTb_dt

        # compressor power
        P_comp = (Q_cool + Q_b_cool) / COP(T_amb)

        # accumulate energy
        energy_Wh += (P_comp * dt) / 3600.0

    return setpoint_time, energy_Wh/1000

In [ ]:
ambient_range = [30, 35, 40, 45]

cooldown_times = []
energy_consumption = []

print("Ambient (°C) | Cooldown Time (min) | Aux Energy (kWh)")
print("-----------------------------------------------")

for Tamb in ambient_range:

    t_set, energy = run_simulation(Tamb)

    cooldown_times.append(t_set)
    energy_consumption.append(energy)

    print(f"{Tamb:>11} | {t_set:>18.2f} | {energy:>14.3f}")

In [ ]:
plt.figure()
plt.plot(ambient_range, cooldown_times, marker='o')
plt.xlabel("Ambient Temperature (°C)")
plt.ylabel("Cabin Cooldown Time (min)")
plt.title("Cooldown Time vs Ambient Temperature")
plt.grid(True)
plt.show()


plt.figure()
plt.plot(ambient_range, energy_consumption, marker='o')
plt.xlabel("Ambient Temperature (°C)")
plt.ylabel("Compressor Energy (kWh)")
plt.title("Auxiliary Energy vs Ambient Temperature")
plt.grid(True)
plt.show()

### Ambient Temperature Sensitivity

The model was evaluated for ambient temperatures of 30°C, 35°C, 40°C, and 45°C while maintaining identical driving and system parameters.

| Ambient Temperature | Cabin Cooldown Time | Compressor Energy |
|---------------------|--------------------|-------------------|
| 30°C | 6.47 min | 0.517 kWh |
| 35°C | 8.70 min | 0.604 kWh |
| 40°C | 11.57 min | 0.710 kWh |
| 45°C | 15.40 min | 0.826 kWh |

Two clear trends emerge:

1. **Cooldown duration increases rapidly with ambient temperature.**

   As ambient temperature rises, the thermal gradient between the cabin and the environment increases, driving higher heat gain into the cabin. At the same time, HVAC system efficiency decreases due to lower compressor COP at higher ambient temperatures.

2. **Auxiliary energy consumption increases significantly even under identical driving conditions.**

   Across the evaluated range, compressor energy increases from **0.517 kWh at 30°C** to **0.826 kWh at 45°C**, representing roughly a **60% increase in auxiliary energy consumption** purely due to environmental conditions.

These results highlight how strongly **thermal environment conditions influence EV energy consumption**, even when propulsion load remains unchanged.

This simple sensitivity study illustrates how transient thermal events such as cabin cooldown can become a meaningful contributor to short-duration auxiliary energy consumption in electric vehicles.